# Seção 3.6 — O que o Random Forest está usando? (importância por grupos de features)

**Por que esta seção existe.** O Random Forest chegou a 100% (com 99 features e também só com as 9 de tempo). Antes de aceitar isso, queremos saber *de onde* vem a informação. Se o desempenho depender de uma única coluna ou de um único grupo, vale desconfiar.

**Duas perguntas diferentes:**
1. **Necessidade:** se eu embaralhar *um grupo inteiro* de features nos dados de validação, o desempenho cai quanto? (importância por permutação em grupos)
2. **Suficiência:** se eu treinar o modelo *só com um grupo*, ele ainda acerta? (modelo de um grupo só)

**Por que em grupos e não coluna a coluna:** colunas muito correlacionadas (por exemplo `std_x` e `pico_x`, ou bandas vizinhas da FFT) dividem a importância entre si, e embaralhar uma só faz as duas parecerem irrelevantes. Embaralhar o grupo todo, com a **mesma permutação** nas colunas dele, mede o efeito real de cada domínio.

**Regras:** só treino e folds por blocos (`X_train`, `y_bin_train`, `cv_folds`). O cofre não é tocado.

**Cuidado ao interpretar:** estes testes dizem *o que o modelo usa*, não se o que ele usa é o defeito físico ou a sessão. Para isso, a sessão nova continua sendo o teste decisivo.

## Definição dos grupos

**O que faz:** separa as 99 colunas em 6 grupos com sentido físico: tempo por eixo (3 grupos), a faixa da rotação 1X (5–15 Hz), as harmônicas baixas (15–105 Hz) e as altas, que incluem a rede elétrica de 120/240 Hz (105–305 Hz). Confere que nenhuma coluna ficou de fora nem repetida.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict

cols_fft = [c for c in X_train.columns if c.startswith('fft_')]


def banda_inicial(c):
    """'fft_015_025Hz_x' -> 15 (início da faixa, em Hz)."""
    return int(c.split('_')[1])


grupos = {
    'Tempo X': ['std_x', 'kurt_x', 'pico_x'],
    'Tempo Y': ['std_y', 'kurt_y', 'pico_y'],
    'Tempo Z': ['std_z', 'kurt_z', 'pico_z'],
    'FFT 5-15 Hz (rotação 1X)': [c for c in cols_fft if banda_inicial(c) == 5],
    'FFT 15-105 Hz (harmônicas baixas)': [c for c in cols_fft if 15 <= banda_inicial(c) < 105],
    'FFT 105-305 Hz (altas e rede)': [c for c in cols_fft if banda_inicial(c) >= 105],
}

todas = [c for cols in grupos.values() for c in cols]
assert sorted(todas) == sorted(X_train.columns), "Algum grupo está faltando ou repetido!"
print({g: len(c) for g, c in grupos.items()})

## Necessidade — importância por permutação em grupos

**O que faz:** para cada fold, treina o Random Forest no treino do fold e, na validação, embaralha um grupo de cada vez (mesma permutação em todas as colunas do grupo, 10 repetições) e mede a **queda** em dois números: F1 macro (geral) e recall da anomalia na Vel 1 (onde a informação é mais sutil).

**Como ler:**
- Queda grande = o modelo depende daquele grupo.
- Queda perto de zero = a informação também está em outro grupo (redundância). Isso é comum aqui e não é defeito.
- O traço preto é a variação entre folds e repetições. Quedas menores que isso são ruído.
- O painel da direita repete o teste para o modelo que usa **só as 9 features de tempo**.

In [ ]:
def recall_vel(y, p, vel, v=1):
    m = (vel == v) & (y == 1)
    return ((p == 1) & m).sum() / m.sum() if m.sum() else np.nan


def importancia_por_grupos(pipe, X, y, vel, folds, grupos, n_rep=10, seed=0):
    rng = np.random.default_rng(seed)
    linhas = []
    for k, (tr, va) in enumerate(folds):
        modelo = clone(pipe).fit(X.iloc[tr], y[tr])
        Xva, yva, vva = X.iloc[va], y[va], vel[va]
        p0 = modelo.predict(Xva)
        base_f1, base_v1 = f1_score(yva, p0, average='macro'), recall_vel(yva, p0, vva)
        for g, cols in grupos.items():
            for _ in range(n_rep):
                Xp = Xva.copy()
                Xp[cols] = Xva[cols].values[rng.permutation(len(Xva))]      # mesma permutação no grupo todo
                p = modelo.predict(Xp)
                linhas.append({'grupo': g, 'fold': k,
                               'Queda no F1 macro': base_f1 - f1_score(yva, p, average='macro'),
                               'Queda no recall Vel 1': base_v1 - recall_vel(yva, p, vva)})
    return pd.DataFrame(linhas)


vel_train = meta_train['velocidade'].values
pipe_rf = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))])
grupos_tempo = {g: c for g, c in grupos.items() if g.startswith('Tempo')}

imp_99 = importancia_por_grupos(pipe_rf, X_train, y_bin_train, vel_train, cv_folds, grupos)
imp_tp = importancia_por_grupos(pipe_rf, X_train[sum(grupos_tempo.values(), [])], y_bin_train, vel_train, cv_folds, grupos_tempo)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.5, 1]})
for ax, imp, titulo in [(axes[0], imp_99, 'Random Forest com 99 features'), (axes[1], imp_tp, 'Random Forest só com as 9 de tempo')]:
    media = imp.groupby('grupo')[['Queda no F1 macro', 'Queda no recall Vel 1']].mean()
    erro = imp.groupby('grupo')[['Queda no F1 macro', 'Queda no recall Vel 1']].std()
    media.plot(kind='barh', xerr=erro, ax=ax, color=['#1f77b4', '#d95f02'], error_kw={'ecolor': 'black', 'lw': 1})
    ax.set_title(titulo, fontweight='bold'); ax.set_xlabel('Queda após embaralhar o grupo'); ax.set_ylabel('')
    ax.grid(True, axis='x', linestyle='--', alpha=.4)
plt.tight_layout(); plt.show()

## Suficiência — um modelo para cada grupo, sozinho

**O que faz:** treina um Random Forest usando *apenas* as colunas de um grupo, com os mesmos folds por blocos, e mostra o recall da anomalia por velocidade e o falso alarme.

**Por quê:** complementa o teste anterior. Um grupo pode não fazer falta (porque outros cobrem) e mesmo assim ser suficiente sozinho, ou o contrário.

**Como ler:**
- Um grupo que sozinho chega perto de 100% em todas as velocidades é suficiente para detectar *dentro da sessão*.
- Se **só** o grupo da rotação 1X (5–15 Hz) já separa a Vel 1, lembre que essa faixa mede a amplitude do 1X: o desbalanceamento a aumenta, mas a diferença de sessão (temperatura, fixação) também pode. Esse é um sinal para priorizar a sessão nova.
- Se os grupos de **tempo** bastam, o modelo pode ser bem mais simples que o de 99 features (e mais barato de calcular na produção), desde que a sessão nova confirme.

In [ ]:
def metricas_grupo(y, p, vel):
    saida = {}
    for v in [1, 2, 3]:
        m = vel == v
        saida[f'Recall Vel {v}'] = ((p == 1) & (y == 1) & m).sum() / ((y == 1) & m).sum()
    saida['FPR total'] = ((p == 1) & (y == 0)).sum() / (y == 0).sum()
    saida['F1 macro'] = f1_score(y, p, average='macro')
    return saida


linhas = {}
for g, cols in grupos.items():
    p = cross_val_predict(pipe_rf, X_train[cols], y_bin_train, cv=cv_folds)
    linhas[g] = metricas_grupo(y_bin_train, p, vel_train)
tab_suf = pd.DataFrame(linhas).T
display(tab_suf.round(3))

plt.figure(figsize=(9, 4))
sns.heatmap(tab_suf[['Recall Vel 1', 'Recall Vel 2', 'Recall Vel 3', 'F1 macro']] * 100, annot=True, fmt='.0f',
            cmap='Blues', vmin=0, vmax=100, cbar=False)
plt.title("Cada grupo sozinho: recall da anomalia por velocidade e F1 (%)", fontweight='bold')
plt.tight_layout(); plt.show()

## Como ler os dois testes juntos

| Padrão | O que indica |
|---|---|
| Quedas pequenas em todos os grupos, e vários grupos suficientes sozinhos | A informação é redundante: o modelo não depende de uma única coluna. Bom sinal de robustez interna, não de generalização |
| Queda grande em um grupo e só ele é suficiente | O modelo depende desse grupo. Investigue o que ele mede (amplitude, rotação, rede elétrica) |
| `FFT 105-305 Hz (altas e rede)` sozinho separa bem | Cuidado: inclui a rede elétrica (120/240 Hz) e ruído. Se separa, pergunte se a sessão difere nesses níveis |
| Só `Tempo` já basta | Modelo mais simples possível; confirmar na sessão nova |

**Lembrete:** nenhum destes testes separa "defeito físico" de "sessão". Eles dizem *o que* o modelo usa. A sessão nova de coleta é o que diz *por que*.